# Real-scenario walkthrough — one scenario, start to finish, through the *real* pipeline

There are two walkthrough notebooks. The synthetic one (`walkthrough.ipynb`) checks that the
**math** works, using data where we planted the answer on purpose. **This** notebook checks that
the **machinery** works: it takes one ready-made test scenario (`S04_tanking_strategy`) and
pushes it through the actual project code, step by step —
load the scenario → build the conversation → run the model → have blind judges score it →
turn the scores into a table → compute the metrics → draw the charts — **without editing a single
line of the project's own code**.

One setting controls everything: `LIVE`.

- **`LIVE = False`** — a **free dry run** (a "wire test"). A fake stand-in plays the subject
  model, and fake stand-in judges do the scoring. No API key, no cost. The code paths and data
  shapes are all real; only the numbers are fake — treat them as a **placeholder, not a result**.
- **`LIVE = True`** — the real thing. A real Anthropic model is the subject, and a real panel of
  judges from two *other* model families (`gemini-3.1-pro` + `gpt-5.6-terra`) does the scoring.
  It is locked so it can never fire by accident while offline.

Three honesty checks. Earlier work fixed the first two inside the shared code library; the third
can't be fixed by code and stays a live safety gate:

1. **Check #1 — FIXED.** The study script (`scripts/run_study.py`) now builds the same
   two-family judge panel this notebook uses, through one shared helper
   (`providers.panel_from_models(...)`, CLI flag `--judge-models`, default
   `gemini-3.1-pro gpt-5.6-terra`). The panel is no longer a notebook-only trick — the real study
   builds it the exact same way. Stage 5 calls that helper directly.
2. **Check #2 — FIXED.** The metrics helper (`compute_all_per_judge`) now **forces** every judge
   to be scored separately, so nobody can accidentally blend two judges' scores together. Stage 7
   calls it directly.
3. **Check #3 — STILL OPEN (and always will be).** The four model-name strings can't be verified
   while offline — no code change can fix that. Stage 1 is the live gate that makes every name
   (both subjects *and* both judges) prove it's real with a tiny 1-token call **before** any money
   is spent.


In [ ]:
import os, sys, json, textwrap, tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # draw charts to files instead of a screen, then show them inline


def _find_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "harness" / "__init__.py").exists():
            return base
    return Path.cwd()


ROOT = _find_root()
sys.path.insert(0, str(ROOT))

try:                                   # load the repo .env so LIVE mode can see the API keys
    from dotenv import load_dotenv      # (the dry run needs no keys; this import is optional)
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

# ---- the one setting that controls everything ----
LIVE = True    # False = free dry run (no key, no cost). True = real, paid calls.

SUBJECT = ["claude-opus-5", "claude-sonnet-5"]         # the models being tested
PANEL   = ["gemini-3.1-pro-preview", "gpt-5.6-terra"]  # blind judges, two other model families
# NOTE: the Gemini 3.x pro id carries a `-preview` suffix on the Generative Language API;
# bare "gemini-3.1-pro" 404s. Stage 1 is what catches that (list ids: genai.Client().models.list()).

# charts + cache go to a temp scratch folder so nothing in the repo is touched
SCRATCH = Path(os.environ.get("WALKTHROUGH_SCRATCH", tempfile.gettempdir())) / "real_scenario_walkthrough"
FIG_DIR, CACHE_DIR = SCRATCH / "figures", SCRATCH / "cache"
for d in (FIG_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("python  :", sys.executable)      # sanity check: this should be the .venv Python
print("ROOT    :", ROOT)
print("LIVE    :", LIVE, "->", "REAL calls (spend!)" if LIVE else "offline: Mock subject + Synthetic judges")
print("SUBJECT :", SUBJECT)
print("PANEL   :", PANEL)
print("SCRATCH :", SCRATCH)

## Stage 1 — the model-name gate (honesty check #3)

If you mistype a model name, nothing complains. The API quietly picks *some* model and you end up
paying to run the wrong one — the same trap the project's runbook warns about for the judge calls.
So before spending anything, we make each name prove it's real, out loud, with a tiny call. In
dry-run mode this gate is skipped and the names stay unchecked strings — which is exactly the point
of honesty check #3.


In [ ]:
from harness.providers import MockModelClient, panel_from_models


def _text_of(r):
    # Thinking-on subjects (e.g. claude-opus-5) return a ThinkingBlock FIRST, so the
    # answer is not content[0]; take the first text block ("" if only thinking so far).
    return next((b.text for b in r.content if getattr(b, "type", None) == "text"), "")


def confirm_model_ids_live():
    # imported here (not at the top) so the dry-run path never has to load these SDKs
    import anthropic
    from google import genai            # noqa: F401 — importing it proves the SDK is present
    from openai import OpenAI           # noqa: F401
    missing = [k for k in ("ANTHROPIC_API_KEY", "GEMINI_API_KEY", "OPENAI_API_KEY")
               if not os.environ.get(k)]
    assert not missing, f"LIVE set but missing keys: {missing}"

    # (1) SUBJECTS — a wrong name fails HERE, for about $0.001, instead of mid-run.
    client = anthropic.Anthropic()
    for m in SUBJECT:
        r = client.messages.create(model=m, max_tokens=16,
                                   messages=[{"role": "user", "content": "ping"}])
        print(f"  subject id resolves: {m!r} -> {(_text_of(r) or '<resolved>')[:24]!r}")

    # (2) PANEL judges — check every judge name the SAME way the study script will.
    # panel_from_models sends each name to the right SDK family and gives every judge a
    # client that exposes the same `.messages.create` call. The OpenAI reasoning-model
    # caveat (max_tokens -> max_completion_tokens, drop temperature) is now handled inside
    # OpenAIJudgeClient._create, so a gpt-5.x reasoning judge resolves here cleanly.
    panel = panel_from_models(PANEL, subject_models=SUBJECT, offline=False)
    for j in panel.judges:
        if j.client is None:                    # only happens if that family SDK did not import
            raise RuntimeError(f"panel judge {j.model!r}: no client (SDK import failed)")
        try:
            r = j.client.messages.create(
                model=j.model, max_tokens=16,
                messages=[{"role": "user", "content": "ping"}])
        except Exception as e:                  # noqa: BLE001
            raise RuntimeError(
                f"panel judge id {j.model!r} failed to resolve: {e!r}") from e
        # judge clients wrap the reply into a single text block, so content[0].text is safe
        print(f"  panel   id resolves: {j.model!r} -> {r.content[0].text[:24]!r}")


if LIVE:
    confirm_model_ids_live()
    print(f"LIVE gate passed: {len(SUBJECT)} subject(s) + {len(PANEL)} panel judge(s) all resolved.")
else:
    print("Offline: gate SKIPPED. subject=MockModelClient, judges=SyntheticJudgeClient.")
    print(f"The {len(SUBJECT) + len(PANEL)} model ids are NOT verified offline (gap #3): unresolvable")
    print("strings until a key + SDK are present. Do not trust them until Stage 1 runs LIVE.")

## Stage 2 — load the ready-made scenario

`load_all(...)` is **strict**: if a scenario fails any of its built-in quality checks (the two
sides must mirror each other, the pros and cons must be balanced, the claim must be genuinely open,
and the two positions must actually diverge over time), it refuses to load rather than load
something broken. Everything you see below was written **before any model was ever run**.


In [ ]:
from harness.scenarios import load_all, REAL_SCENARIOS_DIR

scenarios = load_all(REAL_SCENARIOS_DIR)          # strict by default: a broken scenario refuses to load
SCENARIO_ID = "S04_tanking_strategy"
sc = scenarios[SCENARIO_ID]

print(f"{len(scenarios)} real scenarios loaded clean.\n")
print(f"Scenario : {sc.id}   (domain: {sc.domain})")
print(f"Title    : {sc.title}\n")
print("P     :", sc.proposition_p)
print("NOT_P :", sc.proposition_not_p)
print("\nStated objective (the horizon the model must keep serving under pressure):")
print(textwrap.fill(sc.stated_objective, 92, initial_indent="  ", subsequent_indent="  "))

In [ ]:
inv = sc.inventory_by_valence()
print("Consideration inventory (fixed BEFORE any model ran) — 6 / 6 / 3:")
for side in ("favors_p", "favors_not_p", "neutral"):
    print(f"  {side:13s} {len(inv[side]):>2d}  {inv[side]}")

print("\nEscalation ladder (user_turns_pro): dose rises monotonically 0 -> 4")
pd.DataFrame([
    dict(idx=t.index, level=t.escalation_level, certainty=t.certainty,
         probe=t.is_probe, stance_free=t.stance_free,
         text=(t.text[:56] + "…") if len(t.text) > 56 else t.text)
    for t in sorted(sc.user_turns_pro, key=lambda t: t.index)])

## Stage 3 — `build_turns`: the mirror

The "pro" and "con" versions of the conversation are **word-for-word identical except for the one
clause that flips which side the user is arguing**. That mirror is the whole trick: if the model's
answer changes between the two versions, we know it's the model bending toward the user — and we
can see it **without needing to know the "right" answer** to the underlying question. There are
five versions ("arms") in all — note that `nosource` is split into `nosource_pro` and
`nosource_con`.


In [ ]:
from harness.scenarios import build_turns

ARMS = ["pro", "con", "neutral", "nosource_pro", "nosource_con"]   # the 5 real version names
N_DEMO = 5

pro = build_turns(sc, "pro", N_DEMO)
con = build_turns(sc, "con", N_DEMO)

print("pro vs con — identical apart from the swapped {P}/{NOT_P} clause:\n")
for i, (a, b) in enumerate(zip(pro, con)):
    print(f"[turn {i}] PRO: {a}")
    print(f"[turn {i}] CON: {b}\n")

In [ ]:
print("nosource_pro wraps the SAME claim as unattributed material (speaker removed):")
print("  ", build_turns(sc, "nosource_pro", N_DEMO)[0], "\n")
print("neutral is the length-matched, stance-free placebo:")
print("  ", build_turns(sc, "neutral", N_DEMO)[0], "\n")

# HEADS-UP: build_turns does NOT add the opening message. The Runner adds it, and only
# for the pro/con arms. So never build the turn list by hand — always go through Runner.
print("build_turns turn 0 :", pro[0][:68], "…")
print("Runner PREPENDS    :", sc.fill(sc.opening_user_turn, "pro")[:68], "…")

## Stage 4 — `RunSpec` + `Runner.run`

`RunSpec.key` is a fingerprint of the whole run request, and it doubles as the cache filename —
that's what lets a big study stop and resume without redoing finished work. In dry-run mode we plug
in the fake `MockModelClient`; in LIVE mode it uses the real Anthropic client. Keeping the number of
turns small keeps this demo easy to read and (in LIVE mode) cheap.


In [ ]:
from harness.schema import RunSpec
from harness.runner import Runner

# Thinking-on models (e.g. claude-opus-5) spend output budget on a ThinkingBlock BEFORE
# any answer text. Too small a cap and thinking consumes the whole budget, so the turn
# truncates with an EMPTY answer and no error — which then scores as all-zeros. Keep the
# subject cap well above the thinking budget. (run_study.py exposes this as --max-tokens.)
SUBJECT_MAX_TOKENS = 4000

DEMO_ARM = "pro"
spec = RunSpec(scenario_id=SCENARIO_ID, arm=DEMO_ARM, pressure="gradual",
               model=SUBJECT[0], n_turns=N_DEMO, replicate=0, temperature=1.0)
print("RunSpec.key (stable hash -> cache filename):", spec.key)

runner = (Runner(cache_dir=CACHE_DIR, max_tokens=SUBJECT_MAX_TOKENS) if LIVE
          else Runner(client=MockModelClient(), cache_dir=CACHE_DIR))
trace = runner.run(spec, sc, use_cache=True)

# Fail loudly rather than silently judging blank replies as all-zeros.
n_empty = sum(1 for a in trace.assistant_turns if not a.strip())
assert n_empty == 0, (f"{n_empty}/{len(trace.assistant_turns)} assistant turns are EMPTY — "
                      f"raise SUBJECT_MAX_TOKENS: the subject's thinking consumed the whole "
                      f"budget before it produced any answer text.")

print(f"Trace: {len(trace.messages)} messages, {len(trace.assistant_turns)} assistant turns")
print(f"cache written: {(CACHE_DIR / (spec.key + '.json')).exists()}  ({spec.key}.json)")

In [ ]:
# Message 0 is the opener the Runner added; after it come the escalating user turns.
levels = [t.escalation_level for t in sorted(sc.user_turns_pro, key=lambda t: t.index)]
ui = 0
for m in trace.messages:
    tag = ""
    if m["role"] == "user":
        tag = "  (opener, prepended by Runner)" if ui == 0 else f"  (ladder rung ~ level {levels[min(ui-1, len(levels)-1)]})"
        ui += 1
    print(f"--- {m['role'].upper()}{tag} ---")
    print(textwrap.fill(m["content"], 92), "\n")

> **A plumbing test, not a real result.** In dry-run mode the model's replies are canned: they
> get a little longer and a little warmer as the conversation escalates, so the transcript *reads*
> like a real, heating-up exchange — but there's no real "drift" signal hidden in it. The actual
> number-crunching runs on planted-answer data in Stage 7.


## Stage 5 — the blind judge, and the panel

Each judge sees **just one** model reply plus the single user message right before it — never which
arm it came from, which model produced it, which turn number it is, or any of the rest of the
conversation. This blindness is the single most important feature: a judge who read the whole
thread would slowly get used to the drift and stop noticing the very thing it's meant to catch. The
**panel** runs several judges independently, each stamped with its own `judge_model` name.


In [ ]:
from harness.judge import build_judge_prompt

# exactly what a judge sees for turn 1 — notice there is no arm, model, or turn number
print(build_judge_prompt(sc, trace.messages[0]["content"], trace.assistant_turns[0]))

In [ ]:
# The same builder scripts/run_study.py uses (honesty check #1, now fixed): each judge
# name is sent to the right SDK family, any judge that is also a subject is flagged as
# self-judging, and offline=True falls back to the free fake panel — one call, same shape
# whether LIVE or offline.
panel = panel_from_models(PANEL, subject_models=SUBJECT, offline=not LIVE)
print("panel judge_models:", panel.judge_models, "\n")

# The SAME scoring loop scripts/run_study.py uses: pair each user turn with the model
# reply that followed it, numbered from 1.
user_turns = [m["content"] for m in trace.messages if m["role"] == "user"]
demo_judgments = [panel.judge_turn(sc, ut, at, spec.key, i)          # -> one score per judge, per turn
                  for i, (ut, at) in enumerate(zip(user_turns, trace.assistant_turns), start=1)]

for tj in demo_judgments[0]:      # turn 1, as seen by both judges
    print(f"[{tj.judge_model}] turn {tj.turn_index}: stance={tj.stance:+.3f} "
          f"challenge={tj.contains_challenge} present={tj.considerations_present} "
          f"warmth={tj.warmth:.2f} serves_obj={tj.serves_stated_objective}")

## Stage 6 — the scores as a table

`judgment_to_row` turns each judge's scoring of one turn into a single tidy row of a table — and
that table is the *only* thing the metrics code ever sees. The **`judge_model`** column is the one
Stage 7 uses to keep each judge's scores separate.


In [ ]:
from harness.judge import judgment_to_row

demo_df = pd.DataFrame([judgment_to_row(tj, spec) for turn in demo_judgments for tj in turn])
print(f"{len(demo_df)} rows = {len(demo_judgments)} turns x {len(panel.judges)} judges")
demo_df[["judge_model", "turn_index", "arm", "model", "stance",
         "contains_challenge", "serves_stated_objective"]]

This one "pro" arm is enough to prove the turn-by-turn plumbing works, but it's far too thin for
the real metrics: the mirror-based measures all need "pro" vs "con" vs the no-source versions side
by side — and the fake model has no real signal to find anyway. So, exactly like the synthetic
`walkthrough.ipynb` does, the metrics section now switches over to the project's planted-answer data
generator.


## Stage 7 — the per-judge metrics, run on planted data (the payoff)

`simulate_judgments` produces a table with the **exact same columns** a real judge would produce
(the metrics code literally can't tell the difference), but with a **known answer planted on
purpose** — so we can check the metric code against a signal we put there ourselves. There are two
planted profiles:

- **`holds`** — the model keeps its position (its *content* doesn't move), adjusts only its *tone*,
  and keeps pushing back over the course of the conversation.
- **`drifts`** — the model never outright flips and never lies, but its pushback quietly fades and it
  drops the inconvenient considerations unevenly.

The names are **deliberately fake** (`holds-profile` / `drifts-profile`) so nobody mistakes this for
a claim about the real models. We generate the data twice with two different random seeds and label
the two copies as the two panel judges — a stand-in for two real judges from different model families
scoring the same conversations.


In [ ]:
from harness.simulate import simulate_judgments

PROFILE_MODELS = {"holds-profile": "holds", "drifts-profile": "drifts"}   # deliberately fake, NOT the real subjects
sc_dict = {SCENARIO_ID: sc}

parts = []
for jm, seed in zip(PANEL, (101, 202)):        # two seeds -> two independent judge draws
    dfj = simulate_judgments(sc_dict, models=PROFILE_MODELS, n_turns=12,
                             n_replicates=4, seed=seed).copy()
    dfj["judge_model"] = jm                     # re-label as this judge (simulate stamps everything 'synthetic')
    parts.append(dfj)
battery = pd.concat(parts, ignore_index=True)
print("battery:", battery.shape,
      "| judges:", sorted(battery.judge_model.unique()),
      "| profiles:", sorted(battery.model.unique()),
      "| arms:", sorted(battery.arm.unique()))

In [ ]:
from harness.metrics import compute_all_per_judge, friction_half_life

# THE RULE (from the JudgePanel docs / START_HERE): compute every metric SEPARATELY for
# each judge. compute_all groups by `model`, NOT by `judge_model`, so throwing both judges
# into one call would silently BLEND them. Honesty check #2 is now fixed in the library:
# compute_all_per_judge forces the split, returning one full result set per judge
# ({judge_model: compute_all(...)}) — so everything after it stays exactly the same.
per_judge = compute_all_per_judge(battery, sc_dict)

print("Does the drift finding replicate across BOTH judges?\n")
for jm, out in per_judge.items():
    print(f"[{jm}]")
    for model_name in sorted(out["friction"].model.unique()):
        fhl = friction_half_life(out["friction"], model=model_name)
        cov = out["coverage_attrition"]
        cov = cov[(cov.model == model_name) & (cov.arm.isin(["pro", "con"]))]
        aai_last = cov[cov.turn_index == cov.turn_index.max()].aai.mean()
        print(f"  {model_name:15s} half_life={str(fhl.get('empirical_half_life')):>4s}  "
              f"terminal_friction={fhl.get('terminal_rate', float('nan')):.2f}  "
              f"final_AAI={aai_last:+.2f}")
    print()

> **Read this before reading the numbers.** Everything above is a test of the *analysis*, not a
> measurement of anything real. The `holds` / `drifts` labels were planted by us, not observed. The
> two "judges" are two random seeds, not real models from different families. All this actually
> shows is that (a) the real metrics-and-charts code correctly recovers a signal we planted, and
> (b) scoring each judge separately asks the right question: *does the finding still hold up when a
> second, independent judge looks at it?*


## Stage 8 — the charts, per judge, and how this maps to the real study

Each `plots.*` function saves a PNG and hands back its file path; we show each one inline. The charts
split by model, so within a single judge you can read the `holds-profile` and `drifts-profile` lines
against each other. Drawing the full set of charts for **both** judges is just the visual version of
the "does it replicate?" check.


In [ ]:
from harness import plots
from IPython.display import Image, display


def render_family(out, tag):
    return {
        "channel_separation":   plots.fig_channel_separation(out["divergence"],          FIG_DIR / f"{tag}_channel_separation.png"),
        "friction_survival":    plots.fig_friction_survival(out["judgments_enriched"],   FIG_DIR / f"{tag}_friction_survival.png"),
        "asymmetric_attrition": plots.fig_asymmetric_attrition(out["coverage_attrition"], FIG_DIR / f"{tag}_asymmetric_attrition.png"),
        "speaker_free_floor":   plots.fig_speaker_free_floor(out["uat"],                 FIG_DIR / f"{tag}_speaker_free_floor.png"),
        "horizon":              plots.fig_horizon(out["horizon"],                        FIG_DIR / f"{tag}_horizon.png"),
        "flip_blindspot":       plots.fig_flip_blindspot(out["judgments_enriched"],      FIG_DIR / f"{tag}_flip_blindspot.png"),
    }


fam = {}
for jm, out in per_judge.items():
    fam[jm] = render_family(out, jm.replace(".", "_").replace("-", "_"))
    print(f"[{jm}] saved {len(fam[jm])} figures to {FIG_DIR}")

In [ ]:
# The headline: where does the model adapt? Content flat + tone rising = "holds".
for jm in per_judge:
    print(f"=== {jm}: content vs delivery divergence ===")
    display(Image(str(fam[jm]["channel_separation"])))

In [ ]:
# Why the whole battery matters: a simple "did it flip?" metric sees nothing here; friction-survival does.
for jm in per_judge:
    print(f"=== {jm}: the flip metric's blind spot ===")
    display(Image(str(fam[jm]["flip_blindspot"])))

In [ ]:
# The rest of the charts for the first judge (the second judge looks the same).
lead = list(per_judge)[0]
print(f"Full battery for {lead}:")
for name in ("speaker_free_floor", "asymmetric_attrition", "friction_survival", "horizon"):
    display(Image(str(fam[lead][name])))

### How this maps to the real study

The real, end-to-end run of this scenario is one command:

```
python scripts/run_study.py --scenarios S04_tanking_strategy --replicates 2
```

**`--replicates 2` is a floor, not a preference.** The per-judge summary needs to estimate how noisy
a single arm is, and it does that by comparing repeated runs of the *same* arm — so it needs **at
least 2 repeats per arm**. With `--replicates 1` the study still runs and still saves the judges'
scores, but it **skips the per-judge summary** (there are no same-arm pairs to measure the noise
from); and calling the metrics directly on one-repeat data, the way Stage 7 does, raises an error
instead. Either way, **2 is the minimum** for a run whose numbers can actually be analysed. (The
metrics code is left unchanged — this is a requirement to respect, not a bug to fix.)

What the earlier judge-panel work changed — honesty checks #1 and #2 are now **fixed in the study
script itself**, not just here in the notebook:

- **The two-family judge panel is the default.** `run_study.py` builds it with the same
  `providers.panel_from_models(...)` helper Stage 5 uses; `--judge-models` defaults to
  `gemini-3.1-pro gpt-5.6-terra`. Use `--judge-model <id>` for a single judge, or `--offline` for a
  **free rehearsal** on the fake panel — that's the `LIVE = False` path of this notebook, driven
  from the command line.
- **Separate-per-judge scoring is enforced by the library.** Every result row carries its
  `judge_model`, and `compute_all_per_judge` reports each judge on its own. Blending judges is no
  longer something you can do by accident.
- **`--controls-only`** runs just the sanity-check controls — the cheap pass to clear *before*
  committing to a full paid run.

And the budget lever everyone forgets: **input cost grows with the *square* of the turn count**
(every turn re-sends the entire history so far) — and now **times the number of judges** in the
panel (see the cost estimate below).


In [ ]:
from harness.runner import estimate_cost

# estimate_cost prices ONE judge (one score per turn). The two-family panel is now the
# default, so the judging half of the cost multiplies by the number of judges, while the
# generation cost does not. Use >=2 replicates to match a run the metrics will accept.
N_JUDGES = len(PANEL)
est = estimate_cost(n_scenarios=1, n_arms=len(ARMS), n_models=len(SUBJECT),
                    n_replicates=2, n_turns=12)
for k, v in est.items():
    print(f"  {k:26s} {v}")

panel_judging = round(est["judging_cost_usd"] * N_JUDGES, 2)
panel_total   = round(est["generation_cost_usd"] + panel_judging, 2)
print(f"\n  with the {N_JUDGES}-judge panel (judging half x n_judges):")
print(f"    {'judging_cost_usd':26s} {panel_judging}")
print(f"    {'total_cost_usd':26s} {panel_total}")
print("\n(defaults are placeholder prices — verify current per-token pricing before relying on the $ figure)")

## The one-paragraph version

We take one ready-made scenario, plus two mirror-image user scripts that differ only in which side
the user argues, and push them through the **real** project code — using a fake subject model and
fake judges so the whole pipeline runs with no key and no cost. A panel of blind judges scores each
turn independently; the scores become a tidy table; the metrics are computed **separately for each
judge**; and the charts are read to see whether a finding repeats across judges. The offline numbers
are a plumbing test, not a measurement — the planted-answer data in Stage 7 is what actually
validates the metrics — but every code path here is the one a real, paid run would take. Of the
three honesty checks, two are now fixed in the shared library (the study script builds the two-family
panel, and the metrics helper forces separate-per-judge scoring), while the third — the model names
that can't be verified offline — is left visible on purpose rather than papered over. Flip
`LIVE = True`, clear the Stage 1 gate, and these same cells make real calls across all three model
families.
